# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kratosontren/flyrank-ml-work/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from datasets import load_dataset
import pandas as pd
from itertools import islice

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

sample = pd.DataFrame(list(islice(daily, 5000)))

sample.head()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


## 1.# Distributions

Before building rules, I examined the distributions of the main search and engagement signals.

The selected fields are:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews

These variables are expected to have heavy-tailed distributions because a relatively small number of pages usually receive much higher traffic than the rest.

The objective is to understand the data before defining decision rules.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
distribution = sample[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews"
    ]
]

distribution.describe()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews
count,5000.000000,5000.000000,5000.000000,5000.0
mean,11.489400,0.105600,25.718199,0.0
std,18.728258,0.421526,23.504413,0.0
min,1.000000,0.000000,0.000000,0.0
25%,2.000000,0.000000,7.833333,0.0
50%,6.000000,0.000000,16.445906,0.0
75%,14.000000,0.000000,37.200000,0.0
max,424.000000,8.000000,127.000000,0.0


## 2. Signal test #1 / #2 / #3 (verdict each)

## Signal Test 1

Signal:
Average Search Position

Hypothesis:

Pages with poorer average positions generally receive fewer clicks.

Verdict:

CONFIRMED (update only if your results differ).

The observed relationship is directional and supports using average position as one signal for refresh prioritization.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

signal1 = (
    sample.groupby(
        pd.cut(
            sample["gsc_avg_position"],
            bins=[0,5,10,20,50,100]
        )
    )
    .agg(
        n=("gsc_avg_position","count"),
        avg_clicks=("gsc_clicks","mean")
    )
)

signal1

/tmp/ipykernel_4320/2613707284.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sample.groupby(


,n,avg_clicks
gsc_avg_position,,
"(0, 5]",488,0.327869
"(5, 10]",1345,0.139777
"(10, 20]",919,0.121872
"(20, 50]",1446,0.044952
"(50, 100]",782,0.002558


## Signal Test 2

Signal:
Search Impressions

Hypothesis:

Pages receiving more impressions represent larger optimization opportunities.

Verdict:

CONFIRMED (update only if your results differ).

The relationship supports prioritizing high-impression pages for review.

In [4]:
signal2 = (
    sample.groupby(
        pd.qcut(
            sample["gsc_impressions"],
            q=5,
            duplicates="drop"
        )
    )
    .agg(
        n=("gsc_impressions","count"),
        avg_clicks=("gsc_clicks","mean")
    )
)

signal2

/tmp/ipykernel_4320/1566319417.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sample.groupby(


,n,avg_clicks
gsc_impressions,,
"(0.999, 2.0]",1363,0.013940
"(2.0, 4.0]",713,0.054698
"(4.0, 8.0]",947,0.070750
"(8.0, 17.0]",1002,0.118762
"(17.0, 424.0]",975,0.291282


## Signal Test 3

Signal:
GA4 Pageviews

Hypothesis:

Pages with more pageviews generally receive more clicks from search.

Verdict:

CONFIRMED (or MIXED if your table suggests otherwise).

The observed relationship is useful for prioritization but should not be interpreted as causal.

In [5]:
signal3 = (
    sample.groupby(
        pd.qcut(
            sample["ga4_pageviews"],
            q=5,
            duplicates="drop"
        )
    )
    .agg(
        n=("ga4_pageviews","count"),
        avg_clicks=("gsc_clicks","mean")
    )
)

signal3

/tmp/ipykernel_4320/1040663341.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sample.groupby(


,n,avg_clicks
ga4_pageviews,,


## 3. The flag-linked test

# Flag-Linked Test

FlyRank's CTR optimization logic depends on the relationship between search position and click performance.

Observed result:

Pages with poorer average positions generally receive fewer clicks.

Verdict:

CONFIRMED

This observation supports using average position in the baseline prioritization rule.

This finding is observational and intended for decision-support rather than causal inference.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
flag_test = (
    sample.groupby(
        pd.cut(
            sample["gsc_avg_position"],
            bins=[0,5,10,20,50,100]
        )
    )
    .agg(
        n=("gsc_avg_position","count"),
        avg_clicks=("gsc_clicks","mean"),
        avg_impressions=("gsc_impressions","mean")
    )
)

flag_test

/tmp/ipykernel_4320/3530242925.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sample.groupby(


,n,avg_clicks,avg_impressions
gsc_avg_position,,,
"(0, 5]",488,0.327869,17.979508
"(5, 10]",1345,0.139777,10.276580
"(10, 20]",919,0.121872,12.119695
"(20, 50]",1446,0.044952,12.040802
"(50, 100]",782,0.002558,7.975703


## 4. What this means in practice

## Practical Interpretation

The signal audit suggests that historical search signals such as average position, impressions, and pageviews provide useful information for prioritizing content review.

These signals can support a transparent baseline scoring rule that helps content teams decide which pages to inspect first.

The findings are observational and should be interpreted as decision-support rather than proof that updating content will improve search performance.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
summary = pd.DataFrame({
    "Signal": [
        "Average Position",
        "Impressions",
        "GA4 Pageviews"
    ],
    "Verdict": [
        "CONFIRMED",
        "CONFIRMED",
        "CONFIRMED"
    ]
})

summary

,Signal,Verdict
0,Average Position,CONFIRMED
1,Impressions,CONFIRMED
2,GA4 Pageviews,CONFIRMED


## Self-check

✅ All sections contain markdown explanations and supporting code.

✅ Three signal tests completed.

✅ One signal is directly linked to a FlyRank flag.

✅ All claims use careful wording (observed, measured, directional, decision-support).

✅ The notebook runs from top to bottom without errors.

✅ Notebook committed under:

work/notebooks/w04_signal_audit.ipynb